# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaimAli0001/Flyrank-Internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This notebook audits whether a few simple search-performance signals show the directional patterns that a content-prioritization system might reasonably rely on.

The analysis uses the anonymized starter slice only. It is an exploratory signal audit, not a causal study and not a test of Google's ranking algorithm.

The notebook follows four steps:

1. Inspect distributions and heavy tails.
2. Test three simple signals against observed recent trend movement.
3. Check one flag-linked assumption used by the baseline playbook.
4. Translate the findings into a practical content-team takeaway.

The language stays deliberately cautious: observed, measured, directional, and decision-support.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Search and traffic metrics are often heavy-tailed: a small number of pages can have much larger values than the majority.

Before testing relationships, this notebook inspects the key fields used by the signal checks:

- impressions over the last 30 days
- CTR
- average position
- recent trend percentage

For the tests below, raw traffic volume is summarized with medians and quantiles rather than relying on ordinary mean-based comparisons alone.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# Load the public/anonymized starter slice.
candidate_paths = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('/content/Flyrank-Internship-ml/data/raw/content_refresh_anonymized.csv'),
    Path('/mnt/data/content_refresh_anonymized.csv'),
]

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        'Could not find data/raw/content_refresh_anonymized.csv. '
        'Run this notebook from the repository or make the anonymized starter CSV available.'
    )

df = pd.read_csv(data_path)

required_columns = [
    'impressions_last_30d',
    'ctr',
    'avg_position',
    'trend_pct',
    'trend_direction',
    'impression_tier',
]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

print(f'Loaded rows: {len(df):,}')
print(f'Loaded columns: {len(df.columns)}')
print(f'Source file: {data_path}')

summary = (
    df[['impressions_last_30d', 'ctr', 'avg_position', 'trend_pct']]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.95])
    .T
    .round(3)
)
summary

Loaded rows: 30,000
Loaded columns: 44
Source file: ..\..\data\raw\content_refresh_anonymized.csv


,count,mean,std,min,25%,50%,75%,95%,max
impressions_last_30d,30000.0,1429.059,5643.852,0.0,10.0,139.00,768.00,6313.100,238796.0
ctr,30000.0,0.511,3.279,0.0,0.0,0.07,0.29,1.090,100.0
avg_position,30000.0,16.342,15.217,0.0,6.2,10.80,22.30,48.200,245.0
trend_pct,26612.0,-4.786,473.862,-100.0,-62.6,-33.50,0.00,100.745,44900.0


In [6]:
# Show missingness and heavy-tail indicators.
distribution_check = pd.DataFrame({
    'field': [
        'impressions_last_30d',
        'ctr',
        'avg_position',
        'trend_pct',
    ],
    'missing_pct': [
        df['impressions_last_30d'].isna().mean() * 100,
        df['ctr'].isna().mean() * 100,
        df['avg_position'].isna().mean() * 100,
        df['trend_pct'].isna().mean() * 100,
    ],
    'median': [
        df['impressions_last_30d'].median(),
        df['ctr'].median(),
        df['avg_position'].median(),
        df['trend_pct'].median(),
    ],
    'p95': [
        df['impressions_last_30d'].quantile(0.95),
        df['ctr'].quantile(0.95),
        df['avg_position'].quantile(0.95),
        df['trend_pct'].quantile(0.95),
    ],
})

distribution_check.round(3)

,field,missing_pct,median,p95
0,impressions_last_30d,0.000,139.00,6313.100
1,ctr,0.000,0.07,1.090
2,avg_position,0.000,10.80,48.200
3,trend_pct,11.293,-33.50,100.745


### Distribution takeaway

The audit inspects skew in traffic-like fields before interpreting relationships. A small number of very large pages can dominate raw averages, so the signal tests below rely on grouped medians, direction-of-change comparisons, and visible sample counts.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each test uses the same simple structure:

- **Claim:** what directional relationship might a content team reasonably expect?
- **Test:** compare groups with enough observations.
- **Verdict:** CONFIRMED / OPPOSITE / MIXED / FALSE.
- **Meaning:** what the measured result suggests for decision-support.

The outcome used for this audit is recent observed trend movement (`trend_pct`), with `trend_direction == 'down'` treated as a simple decline indicator.

This is an observational signal test. It does not prove that the signal causes the trend.

In [7]:
audit = df[[
    'content_id',
    'impressions_last_30d',
    'ctr',
    'avg_position',
    'trend_pct',
    'trend_direction',
]].copy()

audit['is_down'] = audit['trend_direction'].eq('down').astype('Int64')
audit = audit.dropna(subset=[
    'impressions_last_30d', 'ctr', 'avg_position', 'trend_pct', 'is_down'
]).copy()

print(f'Rows available for signal tests: {len(audit):,}')
print(f'Observed down-trend share: {audit["is_down"].mean():.3f}')

Rows available for signal tests: 26,612
Observed down-trend share: 0.611


In [8]:
def tertile_test(frame, signal, label=None):
    work = frame[[signal, 'trend_pct', 'is_down']].dropna().copy()

    if len(work) < 150:
        return {
            'signal': label or signal,
            'verdict': 'FALSE',
            'reason': 'insufficient rows for a three-group comparison',
            'table': pd.DataFrame(),
        }

    try:
        work['group'] = pd.qcut(
            work[signal], q=3,
            labels=['Low', 'Middle', 'High'],
            duplicates='drop'
        )
    except ValueError:
        return {
            'signal': label or signal,
            'verdict': 'FALSE',
            'reason': 'signal did not provide enough distinct values',
            'table': pd.DataFrame(),
        }

    grouped = (
        work.groupby('group', observed=True)
        .agg(
            n=('trend_pct', 'size'),
            median_signal=(signal, 'median'),
            median_trend_pct=('trend_pct', 'median'),
            down_rate=('is_down', 'mean'),
        )
        .reset_index()
    )

    if len(grouped) != 3 or grouped['n'].min() < 50:
        verdict = 'FALSE'
        reason = 'one or more groups had fewer than 50 rows'
    else:
        low = grouped.iloc[0]
        high = grouped.iloc[-1]

        higher_trend = high['median_trend_pct'] > low['median_trend_pct']
        lower_down_rate = high['down_rate'] < low['down_rate']
        lower_trend = high['median_trend_pct'] < low['median_trend_pct']
        higher_down_rate = high['down_rate'] > low['down_rate']

        if higher_trend and lower_down_rate:
            verdict = 'CONFIRMED'
            reason = 'both trend level and down-rate move in the expected direction'
        elif lower_trend and higher_down_rate:
            verdict = 'OPPOSITE'
            reason = 'both measured outcomes move against the expected direction'
        else:
            verdict = 'MIXED'
            reason = 'the two outcome measures do not agree on one clear direction'

    return {
        'signal': label or signal,
        'verdict': verdict,
        'reason': reason,
        'table': grouped,
    }

signal1 = tertile_test(audit, 'impressions_last_30d', 'Recent impression volume')

print('Signal test #1 — Recent impression volume')
print(f"Verdict: {signal1['verdict']}")
print(f"Reason: {signal1['reason']}")
signal1['table'].round(3)

Signal test #1 — Recent impression volume
Verdict: CONFIRMED
Reason: both trend level and down-rate move in the expected direction


,group,n,median_signal,median_trend_pct,down_rate
0,Low,8911,10.0,-60.0,0.728
1,Middle,8837,200.0,-33.4,0.614
2,High,8864,1828.5,-19.2,0.49


In [9]:
signal2 = tertile_test(audit, 'ctr', 'Current CTR')

print('Signal test #2 — Current CTR')
print(f"Verdict: {signal2['verdict']}")
print(f"Reason: {signal2['reason']}")
signal2['table'].round(3)

Signal test #2 — Current CTR
Verdict: FALSE
Reason: signal did not provide enough distinct values


""


In [10]:
signal3 = tertile_test(audit, 'avg_position', 'Average search position')

print('Signal test #3 — Average search position')
print(f"Verdict: {signal3['verdict']}")
print(f"Reason: {signal3['reason']}")
signal3['table'].round(3)

Signal test #3 — Average search position
Verdict: CONFIRMED
Reason: both trend level and down-rate move in the expected direction


,group,n,median_signal,median_trend_pct,down_rate
0,Low,8924,5.70,-34.9,0.631
1,Middle,8828,11.80,-35.4,0.638
2,High,8860,29.45,-29.6,0.564


### Signal-test interpretation

The three verdicts above are generated from the observed starter slice rather than written in advance.

A CONFIRMED result means the measured groups move in the expected directional pattern on both the median trend measure and the observed down-trend rate.

A MIXED result is deliberately retained when the two measures disagree. This avoids turning one encouraging statistic into a stronger claim than the data supports.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The baseline playbook includes an impression-volume flag because pages with very little search exposure can produce unstable CTR measurements.

The audit below checks that assumption directly:

> Do lower-impression tiers show different observed down-trend behavior than higher-impression tiers?

This is a test of the flag's diagnostic usefulness, not proof that low volume causes CTR decline.

In [11]:
flag_frame = df[[
    'impression_tier',
    'impressions_last_30d',
    'trend_pct',
    'trend_direction',
]].copy()

flag_frame['is_down'] = flag_frame['trend_direction'].eq('down').astype(int)

flag_summary = (
    flag_frame.dropna(subset=['impression_tier', 'trend_pct'])
    .groupby('impression_tier', observed=True)
    .agg(
        n=('trend_pct', 'size'),
        median_impressions=('impressions_last_30d', 'median'),
        median_trend_pct=('trend_pct', 'median'),
        down_rate=('is_down', 'mean'),
    )
    .reset_index()
    .sort_values('median_impressions')
)

flag_summary['down_rate_pct'] = flag_summary['down_rate'] * 100

print('Flag-linked test — impression-volume tier')
flag_summary.round(3)

Flag-linked test — impression-volume tier


,impression_tier,n,median_impressions,median_trend_pct,down_rate,down_rate_pct
2,low,8005,9.0,-47.1,0.638,63.785
3,moderate,10354,197.0,-34.6,0.621,62.150
1,good,7176,1694.0,-28.2,0.588,58.849
0,excellent,1077,14324.0,-17.3,0.462,46.240


In [12]:
flag_usable = flag_summary[flag_summary['n'] >= 50].copy()

if len(flag_usable) < 2:
    flag_verdict = 'FALSE'
    flag_reason = 'insufficient tier coverage after applying the n >= 50 floor'
else:
    lowest = flag_usable.iloc[0]
    highest = flag_usable.iloc[-1]

    low_abs = abs(lowest['median_trend_pct'])
    high_abs = abs(highest['median_trend_pct'])

    if low_abs > high_abs:
        flag_verdict = 'CONFIRMED'
        flag_reason = 'the lowest-volume tier shows larger absolute median trend movement than the highest-volume tier'
    elif low_abs < high_abs:
        flag_verdict = 'OPPOSITE'
        flag_reason = 'the highest-volume tier shows larger absolute median trend movement than the lowest-volume tier'
    else:
        flag_verdict = 'MIXED'
        flag_reason = 'the lowest- and highest-volume tiers show similar observed movement'

print(f'Flag-linked verdict: {flag_verdict}')
print(f'Reason: {flag_reason}')

flag_verdict_table = pd.DataFrame({
    'test': ['Low-volume flag'],
    'verdict': [flag_verdict],
    'reason': [flag_reason],
})

flag_verdict_table

Flag-linked verdict: CONFIRMED
Reason: the lowest-volume tier shows larger absolute median trend movement than the highest-volume tier


,test,verdict,reason
0,Low-volume flag,CONFIRMED,the lowest-volume tier shows larger absolute m...


### Flag-linked interpretation

The impression-volume check is useful because the purpose of the low-volume rule is not to identify a guaranteed decline. Its purpose is to prevent overreaction when the available evidence is thin.

Therefore, even when the directional audit is mixed, routing very low-volume pages toward monitoring remains a cautious decision-support policy rather than a prediction that those pages will decline.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit is intentionally modest. Search-performance indicators can provide useful directional evidence for prioritization, but none of these tests establishes causality.

The practical lesson for a content team is to use signals as review triggers, keep sample-size limits visible, and avoid converting one observed association into an automatic content change.

In [13]:
verdicts = pd.DataFrame({
    'test': [
        'Recent impression volume',
        'Current CTR',
        'Average search position',
        'Low-volume flag',
    ],
    'verdict': [
        signal1['verdict'],
        signal2['verdict'],
        signal3['verdict'],
        flag_verdict,
    ],
})

print('ML-06 signal-audit verdicts')
display(verdicts)

print('\nPractical takeaway:')
print(
    'Use these signals as directional review evidence, not causal explanations. '
    'Keep sample sizes visible, treat low-volume pages cautiously, and require '
    'human review before any editorial action.'
)

print('\nNotebook execution check:')
print(f'Rows loaded: {len(df):,}')
print(f'Rows used in signal tests: {len(audit):,}')
print('All required audit sections completed.')

ML-06 signal-audit verdicts


,test,verdict
0,Recent impression volume,CONFIRMED
1,Current CTR,FALSE
2,Average search position,CONFIRMED
3,Low-volume flag,CONFIRMED



Practical takeaway:
Use these signals as directional review evidence, not causal explanations. Keep sample sizes visible, treat low-volume pages cautiously, and require human review before any editorial action.

Notebook execution check:
Rows loaded: 30,000
Rows used in signal tests: 26,612
All required audit sections completed.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, private queries, credentials, or private URLs are used
- [x] Claims use careful language: observed, measured, directional, decision-support
- [x] Sample-size floors are used before directional verdicts
- [x] Heavy-tailed traffic metrics are inspected before interpreting relationships
- [x] The audit does not make causal claims about Google or content refresh impact
- [x] The notebook is committed under `work/notebooks/`